# 02 · Task1 fMRI-target localizer

以 MNI 距离定义候选电极，Task1 只验证其 Color–Gray 响应，不反向定义空间目标。

In [1]:
# [Setup]
from pathlib import Path
import sys, ast
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
ROOT=Path('/home/lirui/liulab_project/ieeg/Project_colorieeg_2026'); PIPE=ROOT/'color_cognition_pipeline'/'analyse_0720'
sys.path.insert(0,str(PIPE)); import config
from utils.epochs import load_epochs
from utils.stats import fdr_bh

In [2]:
# [Candidates] 10-mm primary set plus prespecified sensitivity radii
loc=pd.read_excel(ROOT/'processed_data'/'test001'/'test001_ieegloc.xlsx')
channel_col=next(c for c in loc.columns if c.lower() in ('channel','channelname','name'))
def parse_mni(x):
    try: return np.asarray(ast.literal_eval(str(x)),float)
    except Exception: return np.full(3,np.nan)
loc['coord']=loc['MNI'].map(parse_mni)
for side,target in config.FMRI_TARGETS.items():
    loc[f'd_{side}_mm']=loc['coord'].map(lambda p: np.linalg.norm(p-np.asarray(target)) if np.isfinite(p).all() else np.nan)
loc['target_side']=np.where(loc['d_left_mm']<loc['d_right_mm'],'left','right')
loc['target_distance_mm']=loc[['d_left_mm','d_right_mm']].min(axis=1)
candidates=loc[loc.target_distance_mm<=config.PRIMARY_TARGET_RADIUS_MM].copy()
out=config.subject_result_dir('test001')/'localizer'; out.mkdir(parents=True,exist_ok=True)
loc.to_csv(out/'electrode_distance_all.csv',index=False); candidates.to_csv(out/'candidate_electrodes_10mm.csv',index=False)
display(candidates[[channel_col,'target_side','target_distance_mm','MNI']])

,Channel,target_side,target_distance_mm,MNI
31,D2,right,7.294552,"[40.911,-37.529,-12.439]"
32,D3,right,6.264592,"[40.447,-41.394,-11.008]"
33,D4,right,7.726374,"[39.982,-45.260,-9.577]"
109,D3,right,6.264592,"[40.447,-41.394,-11.008]"
110,D4,right,7.726374,"[39.982,-45.260,-9.577]"
119,D4,right,7.726374,"[39.982,-45.260,-9.577]"


In [3]:
# [Task1] Fixed window effect; time-cluster statistics are added after preprocessing parity checks
ep=load_epochs(config.INTERMEDIATE_ROOT/'test001'/'preprocessing'/'task1_erp.npz')
trig=np.char.replace(ep['triggers'].astype(str),'Trigger-In:','')
color=np.isin(trig,['11','21','31','41']); gray=np.isin(trig,['12','22','32','42'])
win=(ep['times_ms']>=100)&(ep['times_ms']<=400)
rows=[]
for ch in candidates[channel_col].drop_duplicates():
    if ch not in ep['channel_names']: continue
    ci=list(ep['channel_names']).index(ch); a=ep['data'][color,ci][:,win].mean(1); b=ep['data'][gray,ci][:,win].mean(1)
    stat,p=mannwhitneyu(a,b,alternative='two-sided'); rows.append({'channel':ch,'color_mean':a.mean(),'gray_mean':b.mean(),'difference':a.mean()-b.mean(),'p':p})
res=pd.DataFrame(rows); res['p_fdr']=fdr_bh(res.p) if len(res) else []
res.to_csv(out/'task1_candidate_fixed_window.csv',index=False); display(res)

,channel,color_mean,gray_mean,difference,p,p_fdr
0,D2,0.329212,0.383833,-0.054621,0.899798,0.899798
1,D3,0.340078,0.053548,0.286530,0.132780,0.398340
2,D4,-0.229157,-0.165818,-0.063339,0.532969,0.799454
